In [10]:
# Import the data 
import pandas as pd 
import os 

# Working directory Set 
os.chdir(r'D:\Mission_Blitzkreig\12.Load_clean_calculate_metrics')

# Load campaign data 
df = pd.read_csv('ooh_campaigns.csv')

print("data loaded")
print(f"Total campaigns :{len(df)}")
print(f"columns: {df.columns.tolist()}")
print("\nFirst 5 campaigns:")
print(df.head())

data loaded
Total campaigns :10
columns: ['campaign_name', 'city', 'format', 'spend', 'impressions', 'clicks', 'conversions', 'revenue']

First 5 campaigns:
                   campaign_name     city       format   spend  impressions  \
0  Dublin City Centre - 48 Sheet   Dublin     48 Sheet  4200.0     980000.0   
1        Cork Motorway - Digital     Cork      Digital     NaN     620000.0   
2     Belfast Roadside - 6 Sheet  Belfast      6 Sheet  1500.0     310000.0   
3     Dublin Airport - Supersite   Dublin    Supersite  8500.0     180000.0   
4      Galway City - Bus Shelter   Galway  Bus Shelter   950.0     180000.0   

   clicks  conversions  revenue  
0    1240           87  18900.0  
1     890           54  11200.0  
2     420           28   5600.0  
3    3100          210  48000.0  
4     210           15   2800.0  


In [11]:

# Check for data quality issues
print("Mission Values:")
print(df.isnull().sum())

# Check for duploicates 
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Check data types
print("\nData Types:")
print(df.dtypes)

# Basic Statistics
print("\nBasic Statistics:")
print(df.describe())


Mission Values:
campaign_name    0
city             0
format           0
spend            1
impressions      1
clicks           0
conversions      0
revenue          1
dtype: int64

Duplicate rows: 0

Data Types:
campaign_name     object
city              object
format            object
spend            float64
impressions      float64
clicks             int64
conversions        int64
revenue          float64
dtype: object

Basic Statistics:
             spend   impressions       clicks  conversions       revenue
count     9.000000  9.000000e+00    10.000000    10.000000      9.000000
mean   3427.777778  5.677778e+05  1115.000000    74.700000  17188.888889
std    2539.329527  3.988978e+05   904.891031    61.528765  14851.552482
min     950.000000  1.800000e+05   210.000000    15.000000   2800.000000
25%    1500.000000  2.600000e+05   477.500000    31.750000   5600.000000
50%    3100.000000  4.500000e+05   935.000000    58.000000  13500.000000
75%    4200.000000  8.100000e+05  1217.5000

In [12]:
# Clean the data 
# Remove any duplicate campaigns 
df = df.drop_duplicates()

# Fill missing values if any 

df['spend'] = df['spend'].fillna(0)
df['revenue']= df['revenue'].fillna(0)

#Ensure numeric columns ar actually numeric
df['impressions'] = pd.to_numeric(df['impressions'],errors='coerce')
df['clicks'] = pd.to_numeric(df['clicks'],errors='coerce')
df['conversions'] = pd.to_numeric(df['conversions'],errors='coerce')
df['spend'] = pd.to_numeric(df['spend'],errors='coerce')
df['revenue'] = pd.to_numeric(df['revenue'],errors='coerce')

print("Data Cleaned")
print(f"Final campaign count :{len(df)}")

Data Cleaned
Final campaign count :10


In [13]:
# Calculate ALL metrics in one go using vectorized operations
df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)
df['cpm'] = (df['spend'] / (df['impressions'] / 1000)).round(2)
df['roi'] = ((df['revenue'] - df['spend']) / df['spend'] * 100).round(2)
df['cvr'] = (df['conversions'] / df['clicks'] * 100).round(2)

# Add profit column
df['profit'] = df['revenue'] - df['spend']

print("✅ Metrics calculated!")
print("\nCampaigns with metrics:")
print(df[['campaign_name', 'city', 'spend', 'revenue', 'roi', 'ctr', 'cpm']].head(10))

✅ Metrics calculated!

Campaigns with metrics:
                     campaign_name       city   spend  revenue     roi   ctr  \
0    Dublin City Centre - 48 Sheet     Dublin  4200.0  18900.0  350.00  0.13   
1          Cork Motorway - Digital       Cork     0.0  11200.0     inf  0.14   
2       Belfast Roadside - 6 Sheet    Belfast  1500.0   5600.0  273.33  0.14   
3       Dublin Airport - Supersite     Dublin  8500.0  48000.0  464.71  1.72   
4        Galway City - Bus Shelter     Galway   950.0   2800.0  194.74  0.12   
5       Limerick Retail - 48 Sheet   Limerick  3100.0  13500.0  335.48   NaN   
6            Dublin DART - 4 Sheet     Dublin  1800.0      0.0 -100.00  0.14   
7              Cork City - Digital       Cork  3400.0  17600.0  417.65  0.14   
8    Belfast City Hall - Supersite    Belfast  6200.0  33000.0  432.26  0.17   
9  Waterford Retail Park - 6 Sheet  Waterford  1200.0   4100.0  241.67  0.12   

     cpm  
0   4.29  
1   0.00  
2   4.84  
3  47.22  
4   5.28  
5    N

In [15]:
# Sort by ROI to find best campaigns
best_campaigns = df.nlargest(5,'roi')[['campaign_name', 'city', 'roi', 'ctr', 'cpm', 'spend']]
print("\nTop 5 Campaigns by ROI:")
print(best_campaigns)

# Find worst campaigns by ROI
worst_campaigns = df.nsmallest(5,'roi')[['campaign_name', 'city', 'roi', 'ctr', 'cpm', 'spend']]
print("\nBottom 5 Campaigns by ROI:")
print(worst_campaigns)

# Summary Statistics 
print("\nSummary Statistics for ROI:")
print(f"Total Spend : ${df['spend'].sum():,.2f}")
print(f"Total Revenue : ${df['revenue'].sum():,.2f}")
print(f"Total Profit : ${df['profit'].sum():,.2f}")
print(f"Average ROI : {df['roi'].mean():.2f}%") 
print(f"Average CTR : {df['ctr'].mean():.2f}%")
print(f"Average CPM : ${df['cpm'].mean():.2f}")


Top 5 Campaigns by ROI:
                   campaign_name     city     roi   ctr    cpm   spend
1        Cork Motorway - Digital     Cork     inf  0.14   0.00     0.0
3     Dublin Airport - Supersite   Dublin  464.71  1.72  47.22  8500.0
8  Belfast City Hall - Supersite  Belfast  432.26  0.17   4.70  6200.0
7            Cork City - Digital     Cork  417.65  0.14   4.20  3400.0
0  Dublin City Centre - 48 Sheet   Dublin  350.00  0.13   4.29  4200.0

Bottom 5 Campaigns by ROI:
                     campaign_name       city     roi   ctr   cpm   spend
6            Dublin DART - 4 Sheet     Dublin -100.00  0.14  4.00  1800.0
4        Galway City - Bus Shelter     Galway  194.74  0.12  5.28   950.0
9  Waterford Retail Park - 6 Sheet  Waterford  241.67  0.12  4.62  1200.0
2       Belfast Roadside - 6 Sheet    Belfast  273.33  0.14  4.84  1500.0
5       Limerick Retail - 48 Sheet   Limerick  335.48   NaN   NaN  3100.0

Summary Statistics for ROI:
Total Spend : $30,850.00
Total Revenue : $154,70

In [18]:
# Performance by city 
city_performance = df.groupby('city').agg({
    'spend':'sum',
    'revenue':'sum',
    'profit':'sum',
    'roi':'mean',
    'ctr':'mean',
    'cpm':'mean'
}).reset_index()
print("\nPerformance by City:")
print(city_performance)

# Recalculate the metrics for each city
city_performance['roi'] = ((city_performance['revenue'] - city_performance['spend']) / city_performance['spend'] * 100).round(2)
city_performance['ctr'] = (city_performance['profit'] / city_performance['spend'] * 100).round(2)

print("\nCity Performance with Recalculated Metrics:")
print(city_performance.sort_values(by='roi', ascending=False))

# Performance by format 
format_performance = df.groupby('format').agg({
    'spend':'sum',
    'revenue':'sum',
    'profit':'sum',
    'roi':'mean',
    'ctr':'mean',
    'cpm':'mean'
}).reset_index()

print("\nPerformance by Format:")
print(format_performance.sort_values(by='roi', ascending=False))



Performance by City:
        city    spend  revenue   profit         roi       ctr        cpm
0    Belfast   7700.0  38600.0  30900.0  352.795000  0.155000   4.770000
1       Cork   3400.0  28800.0  25400.0         inf  0.140000   2.100000
2     Dublin  14500.0  66900.0  52400.0  238.236667  0.663333  18.503333
3     Galway    950.0   2800.0   1850.0  194.740000  0.120000   5.280000
4   Limerick   3100.0  13500.0  10400.0  335.480000       NaN        NaN
5  Waterford   1200.0   4100.0   2900.0  241.670000  0.120000   4.620000

City Performance with Recalculated Metrics:
        city    spend  revenue   profit     roi     ctr        cpm
1       Cork   3400.0  28800.0  25400.0  747.06  747.06   2.100000
0    Belfast   7700.0  38600.0  30900.0  401.30  401.30   4.770000
2     Dublin  14500.0  66900.0  52400.0  361.38  361.38  18.503333
4   Limerick   3100.0  13500.0  10400.0  335.48  335.48        NaN
5  Waterford   1200.0   4100.0   2900.0  241.67  241.67   4.620000
3     Galway    950.

In [19]:
# Python Challenge:
# 1. Find the city with the highest total profit
# 2. Find the format with the best CTR
# 3. Create a summary showing: total campaigns, total spend, total revenue by city
# 4. Print campaigns where ROI > 100% AND CTR > 0.3%

In [21]:
# Challenge 1: City with highest total profit
best_city_profit = city_performance.sort_values(by='profit', ascending=False).iloc[0]
print(f"\nCity with highest total profit:")
print(best_city_profit)


City with highest total profit:
city          Dublin
spend        14500.0
revenue      66900.0
profit       52400.0
roi           361.38
ctr           361.38
cpm        18.503333
Name: 2, dtype: object


In [24]:
# Challenge 2. Find the format with the best CTR
best_ctr_format = format_performance.sort_values(by='ctr', ascending=False).iloc[0]
print(f"\nFormat with best CTR:") 
print(best_ctr_format)


Format with best CTR:
format     Supersite
spend        14700.0
revenue      81000.0
profit       66300.0
roi          448.485
ctr            0.945
cpm            25.96
Name: 5, dtype: object


In [26]:
# 3. Create a summary showing: total campaigns, total spend, total revenue by city
city_summary = df.groupby('city').agg({
    'campaign_name':'count',
    'spend':'sum',
    'revenue':'sum'
}).rename(columns={'campaign_name':'total_campaigns'}).reset_index()
print("\nCity Summary:")
print(city_summary)


City Summary:
        city  total_campaigns    spend  revenue
0    Belfast                2   7700.0  38600.0
1       Cork                2   3400.0  28800.0
2     Dublin                3  14500.0  66900.0
3     Galway                1    950.0   2800.0
4   Limerick                1   3100.0  13500.0
5  Waterford                1   1200.0   4100.0


In [27]:
#  Challenge 4 : Print campaigns where ROI > 100% AND CTR > 0.3%
high_roi_ctr_campaigns = df[(df['roi'] > 100) & (df['ctr'] > 0.3)]
print("\nCampaigns with ROI > 100% AND CTR > 0.3%:")
print(high_roi_ctr_campaigns[['campaign_name', 'city', 'roi', 'ctr']])



Campaigns with ROI > 100% AND CTR > 0.3%:
                campaign_name    city     roi   ctr
3  Dublin Airport - Supersite  Dublin  464.71  1.72
